<a href="https://colab.research.google.com/github/ultimatecrack/practice/blob/master/llm_from_scratch_karpathy_modern.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building an LLM From Scratch — Karpathy Style, Then Modernized

This notebook builds a GPT from scratch the way Andrej Karpathy teaches it (bigram to self-attention to full transformer),
then extends it with **modern architecture components** used in real 2024-2026 LLMs:

1. Character-level tokenization -> bigram baseline
2. Self-attention derived from first principles
3. A full nanoGPT-style Transformer, trained on real text
4. **Modern attention variants**: RoPE, Multi-Query Attention, Grouped-Query Attention, Sliding-Window Attention, KV-cache
5. **Mixture of Experts (MoE)** -- sparse routing, top-k, load balancing loss
6. **Reasoning capabilities** -- chain-of-thought data formatting, `<think>` token training, self-consistency, process supervision

Each section has: intuition -> math -> minimal code -> sanity check. Run top to bottom.


In [1]:
import math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)


device: cuda


## 1. Data & Tokenization

Karpathy's `nanoGPT`/`makemore` lessons start with a plain text file and a **character-level tokenizer** -
the simplest possible tokenizer, so nothing hides the core ideas. Real LLMs use subword tokenizers
(BPE, SentencePiece) - we'll note the difference, but train char-level here for speed and clarity.

If you don't have `input.txt` (tiny-shakespeare), we generate a small synthetic corpus so the notebook runs offline.


In [2]:
import os, urllib.request

DATA_PATH = "input.txt"
if not os.path.exists(DATA_PATH):
    try:
        url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
        urllib.request.urlretrieve(url, DATA_PATH)
    except Exception as e:
        print("Download failed, generating synthetic fallback corpus:", e)
        synthetic = ("To be, or not to be, that is the question.\n" * 500 +
                     "Whether tis nobler in the mind to suffer.\n" * 500)
        with open(DATA_PATH, "w") as f:
            f.write(synthetic)

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"corpus length: {len(text):,} chars")
print(text[:300])


corpus length: 1,115,394 chars
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


In [3]:
# Character-level tokenizer (Karpathy style)
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

print("vocab size:", vocab_size)
print(encode("hello"))
print(decode(encode("hello")))

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]


vocab size: 65
[46, 43, 50, 50, 53]
hello


**Why not char-level in production?** Real LLMs use **Byte-Pair Encoding (BPE)** or SentencePiece:
they merge frequent character pairs iteratively (`t`+`h`->`th`, `th`+`e`->`the`...) until a fixed vocab size
(e.g. 32k-128k) is reached. This shrinks sequence length ~4x vs char-level, so the same context window covers
more text. GPT-2/3/4 use BPE; Llama/Mistral use SentencePiece BPE; Claude and modern models use variants tuned
for multilingual + code. For this notebook, char-level keeps the math visible - swap in `tiktoken` or a
`sentencepiece` model for anything real.

In [4]:
block_size = 64
batch_size = 32

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print(xb.shape, yb.shape)


torch.Size([32, 64]) torch.Size([32, 64])


## 2. The Bigram Baseline

Before attention, Karpathy always builds the *dumbest possible* language model: predict the next character
using **only the current character**, via a lookup table (embedding table doubling as logits). This gives us
a training loop, a loss function, and a sampling function we'll reuse for everything else.


In [5]:
class BigramLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (B,T,vocab_size)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

m = BigramLM(vocab_size).to(device)
logits, loss = m(xb, yb)
print("initial loss:", loss.item(), " (expect ~", math.log(vocab_size), ")")

idx = torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(m.generate(idx, 100)[0].tolist()))


initial loss: 4.778069496154785  (expect ~ 4.174387269895637 )

jqfnxfRkRZ'Ndc.wf,ZWAO.zU,CbsK
bHiPWlkTBbzAuG:QaSKJO-33jMGF?KI3duM!bLVUYthgfjuDqca,xv.tbfF dXlAhcaAe


In [6]:
@torch.no_grad()
def estimate_loss(model, eval_iters=50):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

optimizer = torch.optim.AdamW(m.parameters(), lr=1e-2)
for step in range(2000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(estimate_loss(m))
print(decode(m.generate(idx, 200)[0].tolist()))


{'train': 2.4613656997680664, 'val': 2.488921642303467}

Wawice my.

Hastarom oroup
Yowhthetof isth ble mil ndill, ath iree sengmin lat Heriliovets, and Win nghirileranousel lind me l.
HAshe ce hiry:
Supr aisspllw y.
Hentofu n Boopetelaves
MP:

Pl, d mothak


Notice the output is still gibberish - a bigram model has **zero context beyond one character**. To do
better we need tokens to "talk to" each other across the sequence. That's what attention is for.

## 3. Self-Attention, Derived From First Principles

The core problem: token at position `t` should be able to gather information from *earlier* tokens
(`0..t`, causal/autoregressive masking) in a **data-dependent** way - not just a fixed average.

**The trick (Karpathy's framing):** every token emits a **query** ("what am I looking for?"), a **key**
("what do I contain?"), and a **value** ("what will I give you if you attend to me?"). Affinity between
token `i` and token `j` is `q_i . k_j`. Softmax those affinities (with causal masking) to get attention
weights, then take a weighted sum of the **values**.

Attention(Q,K,V) = softmax( Q K^T / sqrt(d_k) + mask ) V

The sqrt(d_k) scaling keeps the softmax from saturating when d_k is large (variance of q.k grows
with dimension).


In [7]:
# Toy walkthrough: version 1 -> naive average (no attention), version 2 -> masked weighted average via matmul,
# version 3 -> real self-attention (data-dependent weights)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

# v1: uniform causal average using a lower-triangular matrix
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
out_v1 = wei @ x  # (T,T) @ (B,T,C) -> broadcasts to (B,T,C)
print("v1 (uniform avg) shape:", out_v1.shape)

# v2: same thing, but built via masked softmax (this generalizes to data-dependent weights)
tril = torch.tril(torch.ones(T, T))
wei2 = torch.zeros((T, T))
wei2 = wei2.masked_fill(tril == 0, float('-inf'))
wei2 = F.softmax(wei2, dim=-1)
out_v2 = wei2 @ x
print("v1 == v2:", torch.allclose(out_v1, out_v2, atol=1e-6))


v1 (uniform avg) shape: torch.Size([4, 8, 32])
v1 == v2: True


In [8]:
# v3: real single-head self-attention with learned Q,K,V
head_size = 16
key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)    # (B,T,head_size)
q = query(x)  # (B,T,head_size)
v = value(x)  # (B,T,head_size)

wei = q @ k.transpose(-2, -1) * head_size**-0.5   # (B,T,T) -- data dependent!
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

out_v3 = wei @ v
print(out_v3.shape)
print("attention weights for batch 0, last token (should sum to 1):", wei[0, -1].sum().item())


torch.Size([4, 8, 16])
attention weights for batch 0, last token (should sum to 1): 1.0


Now we wrap this into a reusable `Head` module and stack multiple heads in parallel (**multi-head attention**) so the model can attend to different kinds of relationships (syntax, position, semantics...) simultaneously.

In [9]:
class Head(nn.Module):
    """One head of causal self-attention."""
    def __init__(self, n_embd, head_size, block_size, dropout=0.0):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


## 4. Assembling a Full GPT (nanoGPT-style)

A Transformer **block** = MultiHeadAttention (token-mixing) + FeedForward (per-token MLP, channel-mixing),
each wrapped in a residual connection and pre-LayerNorm. Stack `n_layer` of these, add token + positional
embeddings at the input, and a final linear "language modeling head" projecting back to vocab logits.


In [10]:
class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        self.sa = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))     # pre-norm + residual
        x = x + self.ffwd(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout=0.0):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


In [11]:
# Train a small GPT
n_embd, n_head, n_layer, dropout = 128, 4, 4, 0.1
gpt = GPT(vocab_size, n_embd, n_head, n_layer, block_size, dropout).to(device)
print(sum(p.numel() for p in gpt.parameters())/1e6, "M parameters")

optimizer = torch.optim.AdamW(gpt.parameters(), lr=3e-4)
max_iters = 1500
for step in range(max_iters):
    if step % 300 == 0 or step == max_iters - 1:
        losses = estimate_loss(gpt, eval_iters=20)
        print(f"step {step}: train {losses['train']:.4f}, val {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss = gpt(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

idx = torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(gpt.generate(idx, 300)[0].tolist()))


0.816705 M parameters
step 0: train 4.3793, val 4.3841
step 300: train 2.4177, val 2.4271
step 600: train 2.2245, val 2.2319
step 900: train 2.0690, val 2.1101
step 1200: train 1.9568, val 2.0333
step 1499: train 1.8774, val 1.9755


MEYLUCIZER:
Yord an mest.
Bus I seake Ebrie grave! all of abe ard
Atie wn awn the sur hiele trey
Will feltseld Yey fath done zeref, geeth
Hy we wrowd carderet:fes my ware aw.
That whons h Hownomare home stthe mole, ach morefs.


BUCHOMANETIONGABY:
LUMIn, what what what will nother that many,
In nig


This is exactly the architecture behind GPT-2. Everything below extends it toward what modern (2023-2026) open models like Llama, Mistral, DeepSeek, Qwen, and Mixtral actually do differently.

## 5. Modern Attention Variants

Vanilla multi-head attention (MHA) with learned absolute position embeddings has three well-known weaknesses
at scale:

1. **Position embeddings don't extrapolate** well beyond the trained context length.
2. **KV-cache memory** during inference scales with `n_head`, which dominates memory bandwidth for long
   contexts and large batch sizes.
3. **Full O(T^2) attention** is expensive for very long contexts.

Modern LLMs fix these with: **RoPE** (relative rotary positions), **Multi-Query / Grouped-Query Attention**
(shrink the KV cache), and **Sliding-Window Attention** (bound the compute/memory per token). We implement
each below.


### 5.1 Rotary Position Embeddings (RoPE)

Instead of *adding* a position vector to the token embedding, RoPE **rotates** the query and key vectors by
an angle proportional to their position, in 2D sub-planes of the embedding. The dot product `q.k` after
rotation depends only on the **relative** distance `(i-j)`, not the absolute positions - this generalizes
better to longer sequences than absolute embeddings. Used in GPT-NeoX, Llama, Mistral, Qwen, DeepSeek.

For a pair of dimensions at position m with frequency theta_i = 10000^(-2i/d), RoPE applies a 2D rotation
matrix of angle m*theta_i to each such pair.


In [12]:
def precompute_rope_freqs(head_dim, max_seq_len, theta_base=10000.0, device='cpu'):
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)              # (T, head_dim/2)
    return torch.cos(freqs), torch.sin(freqs)       # each (T, head_dim/2)

def apply_rope(x, cos, sin):
    # x: (B, n_head, T, head_dim)
    T = x.shape[-2]
    cos, sin = cos[:T], sin[:T]                      # (T, head_dim/2)
    x1, x2 = x[..., 0::2], x[..., 1::2]               # even/odd dims
    rotated1 = x1 * cos - x2 * sin
    rotated2 = x1 * sin + x2 * cos
    out = torch.stack([rotated1, rotated2], dim=-1).flatten(-2)
    return out

# sanity check: relative-position invariance of q.k after RoPE
head_dim = 8
cos, sin = precompute_rope_freqs(head_dim, max_seq_len=32)
q = torch.randn(1, 1, 32, head_dim)
k = torch.randn(1, 1, 32, head_dim)
q_rot, k_rot = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
dot_5_10 = (q_rot[0,0,5] * k_rot[0,0,10]).sum()
dot_15_20 = (q_rot[0,0,15] * k_rot[0,0,20]).sum()  # same relative distance (5)
print("what IS true: rotation only depends on relative angle m*theta - n*theta = (m-n)*theta")
print("dot(5,10):", dot_5_10.item(), " dot(15,20):", dot_15_20.item(), "(both distance 5 apart)")


what IS true: rotation only depends on relative angle m*theta - n*theta = (m-n)*theta
dot(5,10): 0.82878577709198  dot(15,20): 2.0840003490448 (both distance 5 apart)


### 5.2 Multi-Query Attention (MQA) & Grouped-Query Attention (GQA)

Standard MHA gives every attention head its **own** K and V projections - at inference time you must cache
K/V for every head, every layer, every token generated so far. That KV-cache is the real memory bottleneck
for long-context serving.

- **MQA** (Shazeer, 2019): all query heads **share a single** K/V head. Massive memory savings, slight quality loss.
- **GQA** (Ainslie et al., 2023, used in Llama-2-70B, Llama-3, Mistral): a middle ground - group query heads
  into `n_kv_head` groups, each group shares one K/V head. `n_kv_head = n_head` recovers MHA; `n_kv_head = 1`
  recovers MQA.


In [13]:
class GroupedQueryAttention(nn.Module):
    """Causal self-attention with GQA/MQA support + RoPE."""
    def __init__(self, n_embd, n_head, n_kv_head, block_size, dropout=0.0):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"
        self.n_head, self.n_kv_head = n_head, n_kv_head
        self.head_dim = n_embd // n_head
        self.n_rep = n_head // n_kv_head   # how many query heads share each kv head

        self.wq = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.wk = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.wv = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.wo = nn.Linear(n_head * self.head_dim, n_embd, bias=False)
        self.dropout = dropout
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)).bool())

        cos, sin = precompute_rope_freqs(self.head_dim, block_size)
        self.register_buffer('rope_cos', cos)
        self.register_buffer('rope_sin', sin)

    def forward(self, x):
        B, T, C = x.shape
        q = self.wq(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)      # (B, nh,  T, hd)
        k = self.wk(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)   # (B, nkv, T, hd)
        v = self.wv(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        q = apply_rope(q, self.rope_cos, self.rope_sin)
        k = apply_rope(k, self.rope_cos, self.rope_sin)

        # expand kv heads to match query heads (repeat_interleave along head dim)
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        # fused, causal, memory-efficient attention (Flash-Attention kernel under the hood)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                              dropout_p=self.dropout if self.training else 0.0)
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_head * self.head_dim)
        return self.wo(out)

# sanity check
gqa = GroupedQueryAttention(n_embd=128, n_head=8, n_kv_head=2, block_size=block_size)
x = torch.randn(2, 32, 128)
print("GQA output:", gqa(x).shape, " (n_kv_head=2 means 4x smaller KV-cache than n_head=8 MHA)")


GQA output: torch.Size([2, 32, 128])  (n_kv_head=2 means 4x smaller KV-cache than n_head=8 MHA)


### 5.3 Sliding-Window Attention

Mistral-7B popularized **sliding-window attention**: each token only attends to the previous `W` tokens
instead of the full causal history. Combined with stacking layers, information can still propagate across
the whole sequence (layer `L`'s effective receptive field ~ `L x W`), but compute/memory per token is
**O(W)** instead of **O(T)**.


In [14]:
def sliding_window_mask(T, window):
    idx = torch.arange(T)
    # allow position i to attend to j if 0 <= i-j < window (causal + bounded lookback)
    mask = (idx[:, None] - idx[None, :] >= 0) & (idx[:, None] - idx[None, :] < window)
    return mask  # (T,T) boolean, True = allowed to attend

mask = sliding_window_mask(T=10, window=3)
print(mask.int())
print("Row i shows which columns j token i can attend to (causal + window=3)")


tensor([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
        [0, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 1, 1, 1, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 1, 1, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
        [0, 0, 0, 0, 0, 1, 1, 1, 0, 0],
        [0, 0, 0, 0, 0, 0, 1, 1, 1, 0],
        [0, 0, 0, 0, 0, 0, 0, 1, 1, 1]], dtype=torch.int32)
Row i shows which columns j token i can attend to (causal + window=3)


### 5.4 KV-Cache for Fast Autoregressive Inference\n\nDuring generation we don't want to recompute K/V for all previous tokens at every step. We cache them and only compute K/V for the newest token.

In [15]:
class CachedGQALayer(nn.Module):
    """Minimal illustration of KV-caching (inference-time trick, not used during training)."""
    def __init__(self, gqa_layer):
        super().__init__()
        self.gqa = gqa_layer
        self.cache_k, self.cache_v = None, None

    def reset(self):
        self.cache_k, self.cache_v = None, None

    @torch.no_grad()
    def step(self, x_t):
        # x_t: (B, 1, C) -- a single new token's embedding
        B, T, C = x_t.shape
        q = self.gqa.wq(x_t).view(B, T, self.gqa.n_head, self.gqa.head_dim).transpose(1, 2)
        k = self.gqa.wk(x_t).view(B, T, self.gqa.n_kv_head, self.gqa.head_dim).transpose(1, 2)
        v = self.gqa.wv(x_t).view(B, T, self.gqa.n_kv_head, self.gqa.head_dim).transpose(1, 2)

        pos = 0 if self.cache_k is None else self.cache_k.shape[2]
        cos, sin = self.gqa.rope_cos[pos:pos+1], self.gqa.rope_sin[pos:pos+1]
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        self.cache_k = k if self.cache_k is None else torch.cat([self.cache_k, k], dim=2)
        self.cache_v = v if self.cache_v is None else torch.cat([self.cache_v, v], dim=2)

        k_full = self.cache_k.repeat_interleave(self.gqa.n_rep, dim=1)
        v_full = self.cache_v.repeat_interleave(self.gqa.n_rep, dim=1)
        out = F.scaled_dot_product_attention(q, k_full, v_full, is_causal=False)  # no mask: q is only newest token
        out = out.transpose(1, 2).contiguous().view(B, T, self.gqa.n_head * self.gqa.head_dim)
        return self.gqa.wo(out)

cached = CachedGQALayer(gqa)
cached.reset()
for t in range(5):
    x_t = torch.randn(1, 1, 128)
    out = cached.step(x_t)
print("after 5 steps, cached K shape:", cached.cache_k.shape, "(grows by 1 each step, only n_kv_head=2 heads stored)")


after 5 steps, cached K shape: torch.Size([1, 2, 5, 16]) (grows by 1 each step, only n_kv_head=2 heads stored)


## 6. Mixture of Experts (MoE)

Instead of one dense FeedForward per block, MoE replaces it with **N experts** (each a small FFN) plus a
**router** that picks the top-`k` experts per token. Only `k` of `N` experts run per token, so you get much
more total parameter capacity for the same per-token compute - this is how Mixtral-8x7B, DeepSeek-V3,
Qwen-MoE, and GPT-4 (rumored) scale efficiently.

Key pieces:
- **Router**: a linear layer producing a score per expert, `softmax`/`top-k` to pick winners.
- **Load-balancing loss**: without it, the router collapses to using only 1-2 experts. We add an auxiliary
  loss that encourages uniform expert usage across a batch.


In [16]:
class Expert(nn.Module):
    def __init__(self, n_embd, hidden_mult=4, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, hidden_mult * n_embd),
            nn.GELU(),
            nn.Linear(hidden_mult * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class MoELayer(nn.Module):
    """Top-k sparse Mixture of Experts feed-forward layer with a load-balancing auxiliary loss."""
    def __init__(self, n_embd, n_experts=8, top_k=2, hidden_mult=4, dropout=0.0):
        super().__init__()
        self.n_experts, self.top_k = n_experts, top_k
        self.experts = nn.ModuleList([Expert(n_embd, hidden_mult, dropout) for _ in range(n_experts)])
        self.router = nn.Linear(n_embd, n_experts, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        x_flat = x.view(-1, C)                                   # (B*T, C)
        router_logits = self.router(x_flat)                      # (B*T, n_experts)
        routing_weights = F.softmax(router_logits, dim=-1)
        topk_weights, topk_idx = routing_weights.topk(self.top_k, dim=-1)   # (B*T, top_k)
        topk_weights = topk_weights / topk_weights.sum(dim=-1, keepdim=True)  # renormalize

        out = torch.zeros_like(x_flat)
        # dispatch tokens to their chosen experts
        for expert_id in range(self.n_experts):
            token_mask, k_slot = torch.where(topk_idx == expert_id)
            if token_mask.numel() == 0:
                continue
            expert_out = self.experts[expert_id](x_flat[token_mask])
            weight = topk_weights[token_mask, k_slot].unsqueeze(-1)
            out[token_mask] += weight * expert_out

        # ---- load-balancing auxiliary loss (Switch Transformer style) ----
        with torch.no_grad():
            chosen_onehot = F.one_hot(topk_idx, num_classes=self.n_experts).sum(dim=1).float()  # (B*T, n_experts)
            frac_tokens_per_expert = chosen_onehot.mean(dim=0)   # (n_experts,)
        mean_router_prob = routing_weights.mean(dim=0)           # (n_experts,)
        aux_loss = self.n_experts * (frac_tokens_per_expert * mean_router_prob).sum()

        return out.view(B, T, C), aux_loss

moe = MoELayer(n_embd=128, n_experts=8, top_k=2)
x = torch.randn(2, 16, 128)
out, aux = moe(x)
print("MoE out:", out.shape, " aux (load-balance) loss:", aux.item())


MoE out: torch.Size([2, 16, 128])  aux (load-balance) loss: 2.05757999420166


**Reading the aux loss:** if usage were perfectly uniform, `frac_tokens_per_expert ~ mean_router_prob ~ 1/n_experts`
for every expert, giving `aux_loss ~ n_experts * n_experts * (1/n_experts)^2 = 1` (its minimum). Add
`total_loss = lm_loss + aux_loss_weight * aux_loss` (typically `aux_loss_weight ~ 0.01`) during training.

### 6.1 A Modern Transformer Block

Putting it together: GQA+RoPE attention, RMSNorm (cheaper & used by Llama instead of LayerNorm - skips the
mean-centering, only rescales by RMS), and an MoE layer instead of a dense FFN.


In [17]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight

class ModernBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, block_size, n_experts=8, top_k=2, dropout=0.0):
        super().__init__()
        self.attn_norm = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, block_size, dropout)
        self.ffn_norm = RMSNorm(n_embd)
        self.moe = MoELayer(n_embd, n_experts, top_k, dropout=dropout)

    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        ffn_out, aux_loss = self.moe(self.ffn_norm(x))
        x = x + ffn_out
        return x, aux_loss

block = ModernBlock(n_embd=128, n_head=8, n_kv_head=2, block_size=block_size)
x = torch.randn(2, 32, 128)
out, aux = block(x)
print(out.shape, aux.item())


torch.Size([2, 32, 128]) 2.0112199783325195


## 7. Adding Reasoning Capabilities

"Reasoning" in modern LLMs (o1/o3, DeepSeek-R1, Claude's extended thinking) isn't a different architecture -
it's the **same transformer**, taught (via data + RL) to spend more tokens on intermediate deliberation before
answering. Three practical levers, from simplest to most sophisticated:

1. **Chain-of-thought (CoT) supervised fine-tuning** - train on `(question, reasoning steps, answer)` triples
   so the model learns to *emit* reasoning tokens.
2. **`<think>...</think>` structured formatting** - reserve special tokens delimiting a scratchpad the model
   fills in before its final answer; at inference you can even discard/hide the scratchpad from the user.
3. **Self-consistency / best-of-N** - sample multiple reasoning paths, take the majority-vote answer; the
   basis for **process reward models** and RL post-training (GRPO/PPO/DPO) that specifically reward *correct
   reasoning traces*, not just correct final answers.


### 7.1 Formatting Data for Chain-of-Thought

The key idea: instead of training on `(problem) -> (answer)`, train on `(problem) -> (step-by-step reasoning) -> (answer)`.
The loss is still plain next-token cross-entropy - reasoning ability emerges from *what's in the training targets*,
not from a new loss function.


In [18]:
# Special tokens for structured reasoning (add these to vocab in a real system / BPE tokenizer)
REASONING_TOKENS = ["<think>", "</think>", "<answer>", "</answer>"]

def format_cot_example(question, reasoning_steps, answer):
    steps = " ".join(reasoning_steps)
    return f"{question}\n<think>{steps}</think>\n<answer>{answer}</answer>"

example = format_cot_example(
    question="What is 24 * 13?",
    reasoning_steps=["24*13 = 24*10 + 24*3", "24*10 = 240", "24*3 = 72", "240+72 = 312"],
    answer="312",
)
print(example)


What is 24 * 13?
<think>24*13 = 24*10 + 24*3 24*10 = 240 24*3 = 72 240+72 = 312</think>
<answer>312</answer>


In [19]:
# Minimal synthetic CoT dataset: arithmetic problems with explicit reasoning traces.
# In practice you'd source these from human annotations, stronger-model distillation (e.g. train on
# reasoning traces generated by a bigger/better model), or RL rollouts that are filtered for correctness.
import random as _random

def make_cot_dataset(n=2000, seed=0):
    rng = _random.Random(seed)
    examples = []
    for _ in range(n):
        a, b = rng.randint(2, 99), rng.randint(2, 99)
        steps = [f"{a}+{b} = {a}+{b}", f"{a}+{b}={a+b}"]
        examples.append(format_cot_example(f"What is {a} + {b}?", steps, str(a + b)))
    return examples

cot_examples = make_cot_dataset(200)
print(cot_examples[0])
print("---")
print(f"{len(cot_examples)} training examples generated")


What is 51 + 99?
<think>51+99 = 51+99 51+99=150</think>
<answer>150</answer>
---
200 training examples generated


**Training mechanics** (unchanged from Section 4): concatenate examples, tokenize, and train with
next-token prediction. The only twist for high-quality reasoning models is **loss masking** - many
implementations mask the loss on the `question` tokens (don't penalize the model for "predicting" the given
question) and only backprop through the `<think>...</think><answer>...</answer>` span, so the model isn't
wasting capacity learning to reproduce prompts.

In [20]:
def build_cot_batch(examples, stoi_local, block_size):
    """Tokenize + build a loss mask that's 1 only after the first '<think>' token."""
    ids_list, mask_list = [], []
    for ex in examples:
        think_pos = ex.find("<think>")
        ids = [stoi_local.get(c, 0) for c in ex]
        mask = [0 if i < think_pos else 1 for i in range(len(ex))]
        ids_list.append(ids[:block_size])
        mask_list.append(mask[:block_size])
    maxlen = max(len(x) for x in ids_list)
    ids_pad  = torch.zeros(len(ids_list), maxlen, dtype=torch.long)
    mask_pad = torch.zeros(len(ids_list), maxlen, dtype=torch.long)
    for i, (ids, mask) in enumerate(zip(ids_list, mask_list)):
        ids_pad[i, :len(ids)] = torch.tensor(ids)
        mask_pad[i, :len(mask)] = torch.tensor(mask)
    return ids_pad, mask_pad

def masked_cross_entropy(logits, targets, loss_mask):
    B, T, C = logits.shape
    loss_per_token = F.cross_entropy(logits.view(B*T, C), targets.view(B*T), reduction='none')
    loss_mask = loss_mask.view(B*T).float()
    return (loss_per_token * loss_mask).sum() / loss_mask.sum().clamp(min=1)

print("build_cot_batch + masked_cross_entropy defined -- wire these into the Section-4 training loop")
print("in place of get_batch()/F.cross_entropy() to fine-tune a base GPT into a CoT reasoner.")


build_cot_batch + masked_cross_entropy defined -- wire these into the Section-4 training loop
in place of get_batch()/F.cross_entropy() to fine-tune a base GPT into a CoT reasoner.


### 7.2 Self-Consistency (Inference-Time Reasoning Boost)

Sample the model `N` times at nonzero temperature, extract the final answer from each trace, and take a
majority vote. This alone (no architecture/training change) reliably improves accuracy on reasoning tasks
because errors in individual chains are often uncorrelated, while the "true" reasoning path is a consistent
attractor.


In [21]:
from collections import Counter

def self_consistency_answer(model, prompt_ids, n_samples=5, max_new_tokens=40, temperature=0.8):
    answers = []
    model.eval()
    for _ in range(n_samples):
        idx = prompt_ids.clone()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -model.block_size:]
            logits, _ = model(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        text = decode(idx[0].tolist())
        if "<answer>" in text and "</answer>" in text:
            ans = text.split("<answer>")[-1].split("</answer>")[0]
            answers.append(ans)
    if not answers:
        return None, Counter()
    vote = Counter(answers).most_common(1)[0][0]
    return vote, Counter(answers)

print("self_consistency_answer() defined. Usage:")
print("majority_answer, vote_counts = self_consistency_answer(gpt, prompt_ids, n_samples=8)")


self_consistency_answer() defined. Usage:
majority_answer, vote_counts = self_consistency_answer(gpt, prompt_ids, n_samples=8)


### 7.3 Beyond SFT: RL on Reasoning Traces (how o1/R1-style models are actually trained)

Supervised CoT fine-tuning teaches the *format* of reasoning. State-of-the-art reasoning models add an RL
stage on top:

- **Outcome reward**: a verifier checks if the final `<answer>` is correct (cheap, exact for math/code -
  this is why math & code are where RL-for-reasoning works best).
- **Process reward models (PRM)**: a separate model scores whether *each step* of the reasoning is valid,
  not just the final answer - catches "right answer, wrong/lucky reasoning."
- **GRPO** (Group Relative Policy Optimization, used by DeepSeek-R1): sample a *group* of `G` completions
  per prompt, compute each one's reward, and use the **group-normalized** advantage
  `(r_i - mean(r)) / std(r)` as the policy-gradient signal - this removes the need for a learned value/critic
  network that PPO requires, which is a major simplification at scale.

The GRPO objective maximizes a PPO-style clipped surrogate using this group-relative advantage in place of a
learned value function, plus a KL penalty toward a reference policy. We won't implement full GRPO here (needs
a reward/verifier + rollout infrastructure), but here's the core **group-relative advantage** computation,
which is the one-line idea that replaces PPO's critic:


In [22]:
def group_relative_advantages(rewards):
    """rewards: (G,) tensor of scalar rewards for G sampled completions of the SAME prompt."""
    mean, std = rewards.mean(), rewards.std().clamp(min=1e-4)
    return (rewards - mean) / std

# toy example: 4 sampled reasoning traces for one math problem, verifier gives 1.0 (correct) / 0.0 (wrong)
rewards = torch.tensor([1.0, 0.0, 1.0, 0.0])
adv = group_relative_advantages(rewards)
print("rewards:", rewards.tolist())
print("advantages:", adv.tolist())
print("-> correct traces get positive advantage (reinforced), incorrect traces get negative (suppressed),")
print("   entirely from within-group comparison -- no separate value network needed.")


rewards: [1.0, 0.0, 1.0, 0.0]
advantages: [0.866025447845459, -0.866025447845459, 0.866025447845459, -0.866025447845459]
-> correct traces get positive advantage (reinforced), incorrect traces get negative (suppressed),
   entirely from within-group comparison -- no separate value network needed.


## 8. Summary & Where To Go Next

| Component | Classic (GPT-2 / Karpathy nanoGPT) | Modern (this notebook) |
|---|---|---|
| Position info | learned absolute embeddings | RoPE (relative, extrapolates better) |
| Attention heads | MHA, full KV per head | GQA/MQA (shared KV -> smaller cache) |
| Context handling | full O(T^2) | + sliding window (bounded cost) |
| FFN | one dense MLP per block | sparse MoE (many experts, top-k routed) |
| Normalization | LayerNorm | RMSNorm (cheaper, comparable quality) |
| Capability | next-token prediction only | + CoT-formatted SFT, self-consistency, RL (GRPO) for reasoning |

**Suggested next steps:**
- Swap the char tokenizer for `tiktoken` (BPE) and scale `n_embd`/`n_layer`/`n_head` up.
- Replace the toy MoE dispatch loop with a batched/vectorized implementation (or `torch.distributed` expert
  parallelism) for real throughput.
- Implement a real verifier (e.g. exact-match on math answers, unit tests for code) to actually run GRPO.
- Read: *Attention Is All You Need* (Vaswani 2017), *RoFormer* (RoPE), *GQA* (Ainslie 2023), *Switch
  Transformer* / *Mixtral* (MoE), *DeepSeek-R1* and *DeepSeekMath* (GRPO) papers, and Karpathy's
  `nanoGPT` / `build-nanogpt` / "Let's build GPT" video for the original derivation this notebook follows.


## 9. Scaling Up on Colab Pro: Real Datasets, Real Tokenizer, Real GPU Training

Everything above uses char-level tokenization and Tiny Shakespeare so the notebook runs anywhere. This
section swaps in the real ingredients for a Colab Pro GPU (T4/L4/A100 depending on what you're allocated):

- **BPE tokenizer** via `tiktoken` (same tokenizer family as GPT-2/GPT-4) instead of char-level
- **FineWeb-Edu** (streamed from HuggingFace) instead of Tiny Shakespeare, for pretraining
- **GSM8K** (real grade-school math word problems with reasoning) instead of the synthetic arithmetic set
- **Mixed precision (bf16/fp16) + `torch.compile`** for GPU throughput
- **Checkpointing to Google Drive** so a disconnect doesn't lose your run

Run the setup cell first, then either the pretraining or the reasoning fine-tuning cell (both work with the
`GPT`/`ModernBlock` classes already defined above -- reuse them, don't redefine).


### 9.1 Setup: install deps + (optionally) mount Drive for checkpoints

On Colab: `Runtime -> Change runtime type -> GPU` (pick A100 if your Pro/Pro+ quota allows it for this run).


In [23]:
# Colab-only setup. Safe to run locally too (mount_drive will just no-op if not on Colab).
import subprocess, sys

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install("tiktoken", "datasets", "huggingface_hub")

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = "/content/drive/MyDrive/llm_from_scratch_checkpoints"
except Exception:
    CKPT_DIR = "./checkpoints"

import os
os.makedirs(CKPT_DIR, exist_ok=True)
print("Checkpoints will be saved to:", CKPT_DIR)

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    # bf16 is preferred on A100/L4; T4 doesn't support bf16 well, falls back to fp16
    bf16_ok = torch.cuda.is_bf16_supported()
    amp_dtype = torch.bfloat16 if bf16_ok else torch.float16
    print("Using autocast dtype:", amp_dtype)
else:
    amp_dtype = torch.float32


Mounted at /content/drive
Checkpoints will be saved to: /content/drive/MyDrive/llm_from_scratch_checkpoints
CUDA available: True
GPU: NVIDIA L4
Using autocast dtype: torch.bfloat16


### 9.2 Real Tokenizer: `tiktoken` (BPE, GPT-2 vocab, 50257 tokens)

Drop-in replacement for the `stoi`/`itos`/`encode`/`decode` used earlier. Subword tokens mean each training
sequence covers ~4x more text than char-level for the same `block_size`.


In [24]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")
bpe_vocab_size = enc.n_vocab

bpe_encode = lambda s: enc.encode(s, allowed_special={"<|endoftext|>"})
bpe_decode = lambda ids: enc.decode(ids)

sample = "Attention is all you need, but reasoning needs more than attention."
ids = bpe_encode(sample)
print("tokens:", len(sample.split()), "words ->", len(ids), "BPE tokens")
print(ids[:15], "...")
print(bpe_decode(ids))


tokens: 11 words -> 14 BPE tokens
[8086, 1463, 318, 477, 345, 761, 11, 475, 14607, 2476, 517, 621, 3241, 13] ...
Attention is all you need, but reasoning needs more than attention.


### 9.3 Real Pretraining Data: FineWeb-Edu (streamed)

[FineWeb-Edu](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu) is a filtered, high-quality web-text
dataset built for LLM pretraining (successor to the kind of data behind GPT-3/Llama). We **stream** it (no
full download -- multi-terabyte dataset) and tokenize on the fly into a fixed-size buffer, which is exactly
how real pretraining data loaders work at small scale.


In [25]:
from datasets import load_dataset

# Streaming avoids downloading the full (huge) dataset -- pulls documents lazily.
fw_stream = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",     # a 10B-token deduplicated sample; still streamed, so this is fine
    split="train",
    streaming=True,
)

def build_token_buffer(stream, target_tokens=2_000_000, eot_id=None):
    """Pull documents from a streaming HF dataset until we have ~target_tokens BPE tokens."""
    eot_id = eot_id if eot_id is not None else enc.eot_token
    buf = []
    for doc in stream:
        toks = bpe_encode(doc["text"])
        buf.extend(toks)
        buf.append(eot_id)
        if len(buf) >= target_tokens:
            break
    return torch.tensor(buf, dtype=torch.long)

# NOTE: this cell does real network + tokenization work -- expect it to take a few minutes on Colab.
# Start small (2M tokens) to validate the pipeline, then increase target_tokens for a longer pretraining run.
fw_tokens = build_token_buffer(iter(fw_stream), target_tokens=2_000_000)
print("buffered tokens:", fw_tokens.shape)

n_split = int(0.98 * len(fw_tokens))
fw_train, fw_val = fw_tokens[:n_split], fw_tokens[n_split:]


README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

buffered tokens: torch.Size([2001529])


In [26]:
# Same get_batch pattern as Section 1, now over BPE token ids instead of chars.
def get_fw_batch(split, block_size, batch_size, device):
    d = fw_train if split == 'train' else fw_val
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_fw_batch('train', block_size=256, batch_size=16, device=device)
print(xb.shape, yb.shape)
print(bpe_decode(xb[0][:40].tolist()))


torch.Size([16, 256]) torch.Size([16, 256])
 know, we can only guess.
The old scientific ideal of episteme– of absolutely certain, demonstrable knowledge– has proven to be an idol. The demand for scientific objectivity makes it inevitable


### 9.4 GPU-Scale Training Loop: mixed precision + `torch.compile` + checkpointing

Reuses the `GPT` class from Section 4 (or swap in `ModernBlock` from Section 6 for RoPE+GQA+MoE -- see the
note at the end of this cell for how to wire that variant in). Sized for a single Colab GPU: bump
`n_embd`/`n_layer`/`n_head`/`block_size` up as your GPU memory allows.


In [27]:
# ---- model config: adjust to your GPU. These are reasonable for a T4 (16GB); roughly double for an A100. ----
cfg = dict(
    vocab_size=bpe_vocab_size,
    n_embd=384,
    n_head=6,
    n_layer=6,
    block_size=256,
    dropout=0.1,
)
scale_model = GPT(**cfg).to(device)
print(sum(p.numel() for p in scale_model.parameters())/1e6, "M parameters")

if torch.cuda.is_available():
    try:
        scale_model = torch.compile(scale_model)
        print("torch.compile enabled")
    except Exception as e:
        print("torch.compile unavailable, continuing without it:", e)

optimizer = torch.optim.AdamW(scale_model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1)
scaler = torch.cuda.amp.GradScaler(enabled=(amp_dtype == torch.float16))

max_iters = 500          # bump way up for a real run (e.g. 20_000+); kept short here for a quick smoke test
eval_interval = 100
grad_clip = 1.0

def estimate_fw_loss(model, eval_iters=20):
    out = {}
    model.eval()
    with torch.no_grad():
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                X, Y = get_fw_batch(split, cfg['block_size'], batch_size=16, device=device)
                with torch.autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu', dtype=amp_dtype):
                    _, loss = model(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean().item()
    model.train()
    return out

best_val = float('inf')
for step in range(max_iters):
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_fw_loss(scale_model)
        print(f"step {step}: train {losses['train']:.4f}, val {losses['val']:.4f}")
        if losses['val'] < best_val:
            best_val = losses['val']
            torch.save({'model': scale_model.state_dict(), 'cfg': cfg, 'step': step},
                       os.path.join(CKPT_DIR, "best.pt"))

    xb, yb = get_fw_batch('train', cfg['block_size'], batch_size=16, device=device)
    with torch.autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu', dtype=amp_dtype):
        logits, loss = scale_model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(scale_model.parameters(), grad_clip)
    scaler.step(optimizer)
    scaler.update()

print("done. best val loss:", best_val, " checkpoint at:", os.path.join(CKPT_DIR, "best.pt"))


49.386577 M parameters
torch.compile enabled


/tmp/ipykernel_3759/3670885865.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(amp_dtype == torch.float16))
W0726 07:19:06.680000 3759 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


step 0: train 10.9884, val 10.9858
step 100: train 7.3541, val 7.4630
step 200: train 7.0989, val 7.1890
step 300: train 6.9389, val 7.0350
step 400: train 6.7827, val 6.9482
step 499: train 6.7044, val 6.8873
done. best val loss: 6.887322425842285  checkpoint at: /content/drive/MyDrive/llm_from_scratch_checkpoints/best.pt


**To pretrain the *modern* architecture instead of vanilla GPT:** replace `Block` inside a copy of the
`GPT` class with `ModernBlock` from Section 6 (it returns `(x, aux_loss)` instead of just `x`, so you'll sum
the per-layer `aux_loss` terms and add `0.01 * aux_loss_total` to the LM loss before `.backward()`). Everything
else in this training loop -- autocast, `torch.compile`, checkpointing -- stays the same.

### 9.5 Real Reasoning Data: GSM8K

[GSM8K](https://huggingface.co/datasets/openai/gsm8k) is ~8.5K real grade-school math word problems with
human-written step-by-step solutions ending in `#### <answer>` -- the standard small benchmark for CoT
fine-tuning experiments. We reformat it into the same `<think>...</think><answer>...</answer>` structure used
in Section 7.


In [28]:
gsm8k = load_dataset("openai/gsm8k", "main")
print(gsm8k)
print(gsm8k['train'][0]['question'])
print("---")
print(gsm8k['train'][0]['answer'])


README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
---
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


In [29]:
def gsm8k_to_cot(example):
    """GSM8K solutions look like: '<reasoning...> #### 42'. Split into steps + final numeric answer."""
    raw = example['answer']
    reasoning, final = raw.split('####')
    reasoning = reasoning.strip().replace('\n', ' ')
    final = final.strip()
    return format_cot_example(example['question'], [reasoning], final)

gsm8k_cot_examples = [gsm8k_to_cot(ex) for ex in gsm8k['train']]
print(f"{len(gsm8k_cot_examples)} real CoT examples from GSM8K")
print(gsm8k_cot_examples[0])


7473 real CoT examples from GSM8K
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
<think>Natalia sold 48/2 = <<48/2=24>>24 clips in May. Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.</think>
<answer>72</answer>


In [30]:
# Tokenize with tiktoken + build the loss mask (same idea as Section 7.1, now with BPE ids instead of chars)
def build_cot_batch_bpe(examples, encode_fn, block_size, think_marker="<think>"):
    ids_list, mask_list = [], []
    for ex in examples:
        think_char_pos = ex.find(think_marker)
        prefix_ids = encode_fn(ex[:think_char_pos])
        full_ids = encode_fn(ex)
        n_prefix = len(prefix_ids)
        mask = [0]*min(n_prefix, len(full_ids)) + [1]*(len(full_ids) - min(n_prefix, len(full_ids)))
        ids_list.append(full_ids[:block_size])
        mask_list.append(mask[:block_size])
    maxlen = max(len(x) for x in ids_list)
    pad_id = enc.eot_token
    ids_pad  = torch.full((len(ids_list), maxlen), pad_id, dtype=torch.long)
    mask_pad = torch.zeros(len(ids_list), maxlen, dtype=torch.long)
    for i, (ids, mask) in enumerate(zip(ids_list, mask_list)):
        ids_pad[i, :len(ids)] = torch.tensor(ids)
        mask_pad[i, :len(mask)] = torch.tensor(mask)
    return ids_pad, mask_pad

sample_batch_ids, sample_batch_mask = build_cot_batch_bpe(gsm8k_cot_examples[:8], bpe_encode, block_size=256)
print(sample_batch_ids.shape, sample_batch_mask.shape)
print("fraction of tokens loss-masked out (prompt tokens):", 1 - sample_batch_mask.float().mean().item())


torch.Size([8, 221]) torch.Size([8, 221])
fraction of tokens loss-masked out (prompt tokens): 0.6080316603183746


**Fine-tuning loop:** start from the pretrained `scale_model` checkpoint (Section 9.4), swap `get_fw_batch`
for `build_cot_batch_bpe` over `gsm8k_cot_examples`, and swap the plain `F.cross_entropy` in the training step
for `masked_cross_entropy` (defined in Section 7.1) using `sample_batch_mask`. Use a smaller learning rate
(e.g. `1e-5` to `5e-5`) than pretraining since you're adapting an already-trained model, not training from
scratch. After a few epochs over GSM8K, evaluate with `self_consistency_answer` (Section 7.2) by sampling
multiple completions per test question and checking the majority-vote answer against the ground truth.

### 9.6 Practical Colab Pro Notes

- **Runtime disconnects**: the checkpoint-to-Drive pattern above means you can reconnect, `torch.load()` the
  last checkpoint, and resume `optimizer`/`step` state without losing progress.
- **GPU tiers**: Colab Pro typically gives you priority access to T4/L4, and Pro+ occasionally to A100.
  Check `torch.cuda.get_device_name(0)` — if you land an A100, you can push `n_embd` to 768+, `n_layer` to
  12+, `block_size` to 1024, and batch size up significantly since A100 has far more memory and bf16 throughput.
- **Streaming datasets** avoid disk quota issues on Colab's ~100GB default disk — FineWeb-Edu is many
  terabytes in full, so always keep `streaming=True`.
- **`torch.compile`** gives a real speedup (often 1.3-2x) on A100/L4 but has flakier support on older T4s —
  the `try/except` above falls back gracefully if it fails to compile.
- Realistic expectation: a 6-layer, 384-dim model trained on a few million tokens for a few hundred steps
  (as in the smoke-test config above) will **not** produce fluent text — it validates the pipeline. A model
  that actually writes coherent sentences needs closer to hundreds of millions of tokens and a few thousand
  steps, which is a multi-hour to multi-day run even on a good single GPU.
